# 00 — Quick Start: Smoke Test do Stack Completo

Este notebook valida que toda a infraestrutura está funcionando:

| Componente | Teste |
|------------|-------|
| Qdrant | Conectar, criar collection, inserir, buscar |
| sentence-transformers | Criar embeddings |
| Ollama | Gerar resposta com LLM |
| RAG completo | Pipeline end-to-end |

**Pré-requisitos:**
```bash
docker compose up -d
docker exec ollama ollama pull llama3.2
uv sync
```

## 1. Verificar conexões

In [ ]:
import sys
import httpx

print(f"Python: {sys.version}")

# Verificar Qdrant
try:
    r = httpx.get("http://localhost:6333/")
    data = r.json()
    print(f"✅ Qdrant: {data.get('title', 'OK')} v{data.get('version', '?')}")
except Exception as e:
    print(f"❌ Qdrant não está rodando: {e}")
    print("   Execute: docker compose up -d qdrant")

# Verificar Ollama
try:
    r = httpx.get("http://localhost:11434/api/tags")
    models = [m['name'] for m in r.json().get('models', [])]
    print(f"✅ Ollama: modelos disponíveis = {models if models else '(nenhum — execute pull_models.sh)'}")
except Exception as e:
    print(f"❌ Ollama não está rodando: {e}")
    print("   Execute: docker compose up -d ollama")

## 2. Criar Embeddings com sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Carrega modelo (faz download automático ~22MB na primeira vez)
model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "O gato dorme no sofá.",
    "O felino repousa no divã.",       # semanticamente similar à anterior
    "A economia cresceu 3% este ano.",  # semanticamente diferente
    "Machine learning é o futuro.",
    "Inteligência artificial vai transformar o mundo.",  # similar à anterior
]

embeddings = model.encode(sentences, normalize_embeddings=True)

print(f"✅ Embeddings criados!")
print(f"   Shape: {embeddings.shape}  ({len(sentences)} frases × {embeddings.shape[1]} dimensões)")
print(f"   Dtype: {embeddings.dtype}")
print(f"   Memória: {embeddings.nbytes / 1024:.1f} KB")

In [ ]:
# Verificar similaridade semântica
sims = embeddings @ embeddings.T

print("Matriz de similaridade cosine:\n")
import pandas as pd

labels = [s[:30] + "..." for s in sentences]
df = pd.DataFrame(sims, index=labels, columns=labels)
print(df.round(3).to_string())

print(f"\n🔍 'gato dorme' vs 'felino repousa':  {sims[0,1]:.3f} (deve ser alto!)")
print(f"🔍 'gato dorme' vs 'economia':         {sims[0,2]:.3f} (deve ser baixo!)")
print(f"🔍 'machine learning' vs 'IA futuro':  {sims[3,4]:.3f} (deve ser alto!)")

## 3. Indexar no Qdrant

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient(host="localhost", port=6333)

COLLECTION = "quickstart_test"

# Recriar collection
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

# Inserir pontos
points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={"text": sentence, "language": "pt"},
    )
    for i, sentence in enumerate(sentences)
]

client.upsert(collection_name=COLLECTION, points=points)

info = client.get_collection(COLLECTION)
print(f"✅ Qdrant collection '{COLLECTION}' criada")
print(f"   Pontos indexados: {info.points_count}")

In [ ]:
# Busca semântica
query = "animais que dormem"
query_vec = model.encode(query, normalize_embeddings=True)

results = client.query_points(
    collection_name=COLLECTION,
    query=query_vec.tolist(),
    limit=3,
    with_payload=True,
).points

print(f"🔍 Query: '{query}'")
print(f"\nTop {len(results)} resultados:")
for r in results:
    print(f"  Score: {r.score:.3f} | {r.payload['text']}")

## 4. Gerar resposta com Ollama

In [ ]:
import ollama

# Teste simples
response = ollama.chat(
    model="llama3.2",
    messages=[{"role": "user", "content": "Diga 'Hello from Ollama!' em português, em uma frase."}],
)

print(f"✅ Ollama respondeu:")
print(f"   {response['message']['content']}")

## 5. Pipeline RAG end-to-end

In [ ]:
# Mini RAG completo num único bloco
user_question = "Quais animais dormem mencionados nos documentos?"

# 1. Embed query
query_embedding = model.encode(user_question, normalize_embeddings=True)

# 2. Retrieve
search_results = client.query_points(
    collection_name=COLLECTION,
    query=query_embedding.tolist(),
    limit=3,
    with_payload=True,
).points

context = "\n".join(
    f"- {r.payload['text']}" for r in search_results
)

# 3. Generate
rag_prompt = f"""Responda com base APENAS no contexto abaixo.

Contexto:
{context}

Pergunta: {user_question}

Resposta:"""

rag_response = ollama.chat(
    model="llama3.2",
    messages=[{"role": "user", "content": rag_prompt}],
)

print("=" * 60)
print(f"Pergunta: {user_question}")
print("=" * 60)
print(f"\nContexto recuperado:\n{context}")
print(f"\nResposta RAG:\n{rag_response['message']['content']}")
print("=" * 60)
print("\n✅ Stack completo funcionando!")

## Resumo

| Componente | Status |
|-----------|--------|
| sentence-transformers | ✅ Embeddings criados |
| Qdrant | ✅ Indexação e busca funcionando |
| Ollama | ✅ LLM respondendo |
| Pipeline RAG | ✅ Completo |

## Próximos passos

1. **Entender embeddings em profundidade** → [`01_embeddings/01_what_are_embeddings.ipynb`](01_embeddings/01_what_are_embeddings.ipynb)
2. **Dimensões e float types** → `01_embeddings/02_vector_dimensions.ipynb`
3. **RAG pipeline completo** → `03_rag_fundamentals/01_naive_rag.ipynb`